# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [1]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [4]:
dataset["train"][777]

{'label': 0,
 'text': "I loved it, parking and all, after all, they had Palak Paneer! Mmmmm. It was tasty. I spent a lot, but I was ok with the occasional splurge. the second trip, i again bought my hit foods Palak Paneer and it had obviously gone over. \\n\\nEver have whoopsies after green Indian cuisine?\\n\\nIt's off my list now. Forever."}

In [5]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [6]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [7]:
show_random_elements(dataset["train"])

,label,text
0,5 stars,"I actually liked the sundries store... They have everything and a well stocked pharmarcy.. Prices are inflated? It is everywhere, not just that store...."
1,3 stars,"I brought a co worker in for lunch before driving back to So Cal. The place was dead BC of a fatal car accident which closed off access from North Decatur.\n\nOur order - won ton soup, Kung pao beef and orange chicken. The won ton soup came out first but the broth was to dark with soy sauce. The won ton was tasty and full of shrimp and included Chinese greens. When our other entrees came out the chef made Mongolian beef instead of Kung pao which I ended up accepting since we were in a hurry. The orange chicken was decent tasting nothing spectacular.\n\nThe waitress was good although I ended up down grading Tasty Station a star due the the mishaps on the won ton soup and mis ordering."
2,1 star,"When you're here, you're ignored. \n\nA friend and I decided to dine at the bar seating area for the soup/salad/breadsticks special for a late lunch, but what should have been a simple routine order turned into a train wreck that evolved into the last time I eat at OG. \n\nThey had employees (including what appeared to be a manager) wandering aimlessly like malfunctioning roomba vacuums, several tables were left un-bussed for almost an hour, and a complete lack of service even at the basic level left us mind boggled. We watched a couple looking around, waiting for a server, but they did what we should have and walked out after several minutes. Despite knowing what we wanted before we arrived, our order wasn't taken until over a half hour after walking through the door. The salad was under dressed, the breadsticks salty, the soup was fine although the lack of service left a poor taste in our mouths throughout the entire meal. Our server disappeared more often than David Copperfield so locating someone to get a check took several minutes after finishing. The manager finally came over and presented the check but then the train derailed. We immediately put two cards down to pay and waited, and waited, and waited. We looked around, made comments, contemplated just leaving. Nobody wanted to take our money. Finally, 15 minutes later our server reappeared from her disappearing act to run our cards we requested to be split. Obviously too difficult of a task as the entire bill was charged to one card. \n\nI regret leaving any tip at all, but the biggest tip I can leave is avoid this place and find a nice local family restaurant that will appreciate your business. There are plenty of eateries out there that will make you feel like family."
3,5 stars,Eexelent store and very helpful owners. We went there on our way back to LA and picked up a few items. Everything was top quality.
4,3 stars,"Tried KoMex a few weeks ago since my friend, who resides in North Las Vegas was raving about it. I went with the Korean Kogi tacos, which seemed like a popular hit. \n\nThey were alright. I don't want to say they were extremely tasty and at the same time say anything bad. I wish I could try some of their other items since they offer a variety of different fusion type foods. \n\nPlace is really busy though. Try to come at an off time if you can't stand waits."
5,1 star,"Go anywhere else in the Bellagio if you want good service. We tried eating at the bar, waited over 40 minutes for 6 pieces of sushi to show up. The waitress forgot to bring our second round of drinks, we ended up cancelling the rest of our order... the food quality is overrated. The man sitting beside me left because he didn't want to suffer the same fate. Furthermore, the food service manager had nothing to say after we advised him of what had transpired. I find this unacceptable.\n\nWe travel extensively and dine frequently. I highly recommend hitting up Olives or Michael Mina if you want real food AND good service."
6,1 star,i ordered the orange chicken and it was just gross. im no expert on westernized chines

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [10]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,4 stars,"This restaurant has changed hands a few times over the years, but I hope this one sticks, because it's lovely. The owners have form, having come from the successful 'Mya' Indian / Thai restaurant down the road on Commercial street. Although the decor is (for now) largely unchanged from the building's previous incarnation as Amani, the kitchen and other facilities have been completely replaced - a move which suggests the new owners have their priorities right.\n\nThis seems to be confirmed by the quality of the food, which was splendid on the night I went. Both the Indian and Thai sides of the equation stood up to scrutiny, in terms of the ingredients and preparation. The Tom Yum Gai starter was spot-on: delicate and fresh, but packed with flavour to line you up for the main course. The only slight (very slight) let down was the chicken panang, which was a tiny bit insipid.\n\nThe service was really great - very friendly and nice chat. The owners don't drink, so after the meal we were all invited to try a couple of new wines they were considering adding to the list (which is, incidentally, pretty impressive).\n\nI'll definitely go back soon - and take a few more people, so I can try out more of the menu!","[101, 1188, 4382, 1144, 2014, 1493, 170, 1374, 1551, 1166, 1103, 1201, 117, 1133, 146, 2810, 1142, 1141, 13963, 117, 1272, 1122, 112, 188, 9020, 119, 1109, 5032, 1138, 1532, 117, 1515, 1435, 1121, 1103, 2265, 112, 1422, 1161, 112, 1890, 120, 6888, 4382, 1205, 1103, 1812, 1113, 10472, 2472, 119, 1966, 1103, 1260, 19248, 1110, 113, 1111, 1208, 114, 3494, 16684, 1121, 1103, 1459, 112, 188, 2166, 17825, 1112, 7277, 7192, 117, 1103, 3119, 1105, 1168, 3380, 1138, 1151, 2423, 2125, 118, 170, 1815, 1134, 5401, 1103, 1207, 5032, 1138, 1147, 21249, 1268, 119, 165, 183, 165, 183, 1942, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [11]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

### num_labels是什么

把 BERT 预训练模型想象成一台通用机器，它很会“理解语言”，但不会分类。AutoModelForSequenceClassification 是在这台机器上装一个控制面板。
- num_labels 就是面板上有几个按钮。
- num_labels=5 → 装 5 个按钮的面板，每个按钮对应一个类别。
- num_labels=2 → 装 2 个按钮的面板，做“是/否”判断。
按钮的接线（分类头权重）是新装的、随机初始化的，需要你用训练数据教它每个按钮该在什么情况下亮。

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [13]:
from transformers import TrainingArguments

model_dir = "models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  per_device_train_batch_size=16,
                                  num_train_epochs=5,
                                  logging_steps=100)

In [15]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_better=None,
group_by_le

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [16]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

Trainer 是监考老师，compute_metrics 是阅卷规则
- 考试结束（评估阶段），老师收上来一堆答题卡（logits）和标准答案（labels）。
- 老师自己只会算“扣了多少分”（loss），但不会算“正确率是多少”。
- 你给老师一份阅卷规则（compute_metrics）：
- 每道题选答案里分数最高的那个（argmax）；
- 和标准答案对比（metric.compute）；
- 统计正确率，填到成绩单上（返回字典）。
- 老师按你的规则批完，把“正确率 85%”写进成绩单（日志）。

In [18]:
# metric正是上步定义的，python语法
def compute_metrics(eval_pred):
    # logits：模型对每个样本、每个类别的原始分数
    # labels：真实标签（整数索引）
    logits, labels = eval_pred
    # logits → 类别索引
    predictions = np.argmax(logits, axis=-1)
    # 算指标
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

#### evaluation_strategy参数
- "no"	训练中不评估
- "steps"	每 N 个 step 评估一次
- "epoch"	每个 epoch 评估一次

#### 评估太频繁有什么代价？
- 评估不是免费的：
    - 要在验证集上前向传播一遍，耗时。
    - 验证集大时，评估时间可能接近甚至超过训练一个 epoch。
    - 频繁评估会拖慢训练。
- 结论
    - 小数据集、小模型：每个 epoch 评估问题不大。
    - 大数据集、大模型：评估一次很贵，建议降低频率，比如每几百 step 或每几个 epoch 评估一次。
    - 总 epoch 很多时，更没必要每个 epoch 都评估。

In [20]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch",     # 每个 epoch 结束评估一次
                                  per_device_train_batch_size=16,  # 每个设备的训练批次大小 16
                                  num_train_epochs=3,              # 总共训练 3 个 epoch
                                  logging_steps=30)                # 每 30 个 step 记录一次训练日志

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 1 nvidia-smi`:

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 14:37:41 2023

Wed Dec 20 14:37:41 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   64C    P0              69W /  70W |   6665MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     18395      C   /root/miniconda3/bin/python                6660MiB |
+---------------------------------------------------------------------------------------+
```

### 训练参数解析
TrainOutput(
    global_step=189,
    training_loss=1.0374584450292839,
    metrics={
        'train_runtime': 75.5726,
        'train_samples_per_second': 39.697,
        'train_steps_per_second': 2.501,
        'total_flos': 789354427392000.0,
        'train_loss': 1.0374584450292839,
        'epoch': 3.0
    }
)

| 参数 | 你的值 | 含义 |
|---|---|---|
| `global_step` | `189` | 整个训练过程中**参数更新的总次数**。每处理一个 batch 并反向传播一次，step +1。 |
| `training_loss` | `1.0375` | 训练结束时的**平均训练损失**。通常是对所有 step 的 loss 做平均，不是最后一个 batch 的 loss。 |
| `train_runtime` | `75.5726` | 总训练耗时，单位**秒**。约 1 分 16 秒。 |
| `train_samples_per_second` | `39.697` | 训练吞吐量：**每秒处理多少条样本**。 |
| `train_steps_per_second` | `2.501` | 训练速度：**每秒执行多少个 step**。 |
| `total_flos` | `7.8935e14` | 整个训练消耗的**总浮点运算次数**（FLOPs），衡量计算量。 |
| `train_loss` | `1.0375` | 和 `training_loss` 相同，训练损失。 |
| `epoch` | `3.0` | 实际完成的训练轮数，和你设置的 `num_train_epochs=3` 一致。 |

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.380700,1.121430,0.529000
2,0.959800,0.960890,0.587000
3,0.682900,0.974738,0.602000


TrainOutput(global_step=189, training_loss=1.0374584450292839, metrics={'train_runtime': 75.5726, 'train_samples_per_second': 39.697, 'train_steps_per_second': 2.501, 'total_flos': 789354427392000.0, 'train_loss': 1.0374584450292839, 'epoch': 3.0})

In [23]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [24]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 0.9949256181716919,
 'eval_accuracy': 0.55,
 'eval_runtime': 1.0352,
 'eval_samples_per_second': 96.602,
 'eval_steps_per_second': 12.558,
 'epoch': 3.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [25]:
trainer.save_model(model_dir)

In [26]:
trainer.save_state()

In [23]:
# trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少

In [1]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [3]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML
from transformers import AutoTokenizer

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
random_seed = 42

full_split = tokenized_datasets["train"].train_test_split(
    test_size=0.02,
    seed=random_seed,
    stratify_by_column="label",
)

train_dataset = full_split["train"]
eval_dataset = full_split["test"]
test_dataset = tokenized_datasets["test"]  # 最终只评估一次

print("Full train:", len(train_dataset))
print("Validation:", len(eval_dataset))
print("Final test:", len(test_dataset))

Full train: 637000
Validation: 13000
Final test: 50000


In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )

In [7]:
from transformers import TrainingArguments, Trainer

model_dir = "models/bert-base-cased-finetune-yelp-full"

training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch",     # 每个 epoch 结束评估一次
                                  per_device_train_batch_size=8,   # 每个设备的训练批次大小 8
                                  num_train_epochs=3,              # 总共训练 3 个 epoch
                                  logging_steps=30,                # 每 30 个 step 记录一次训练日志
                                  
                                  load_best_model_at_end=True,        # 核心：训练结束加载最佳模型
                                  metric_for_best_model="eval_loss",  # 根据 eval_loss 判断是否为最佳模型
                                  save_strategy="epoch",              # 与评估策略保持一致，每个 epoch 存一次
                                  save_total_limit=2,                 # 最多只保留2个 checkpoint，节省磁盘
                                  bf16=True)                          # BF16 减少显存并提升吞吐

In [8]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [9]:
train_result = trainer.train()

print("Train result:")
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.854800,0.869268,0.629769
2,0.792200,0.816471,0.658846
3,0.769800,0.792456,0.671692


Train result:
TrainOutput(global_step=238875, training_loss=0.8184419561093669, metrics={'train_runtime': 25176.9223, 'train_samples_per_second': 75.903, 'train_steps_per_second': 9.488, 'total_flos': 5.02818770248704e+17, 'train_loss': 0.8184419561093669, 'epoch': 3.0})


In [10]:
print("Train result:")
print(train_result)

Train result:
TrainOutput(global_step=238875, training_loss=0.8184419561093669, metrics={'train_runtime': 25176.9223, 'train_samples_per_second': 75.903, 'train_steps_per_second': 9.488, 'total_flos': 5.02818770248704e+17, 'train_loss': 0.8184419561093669, 'epoch': 3.0})


In [11]:
# ============================================================
# 12. 查看每个 epoch 的验证损失和准确率
# ============================================================
print("\nEpoch metrics")

for item in trainer.state.log_history:
    if "eval_loss" in item:
        print(
            f"epoch={item['epoch']:.2f}",
            f"step={item['step']}",
            f"eval_loss={item['eval_loss']:.6f}",
            f"eval_accuracy={item['eval_accuracy']:.6f}",
        )


Epoch metrics
epoch=1.00 step=79625 eval_loss=0.869268 eval_accuracy=0.629769
epoch=2.00 step=159250 eval_loss=0.816471 eval_accuracy=0.658846
epoch=3.00 step=238875 eval_loss=0.792456 eval_accuracy=0.671692


In [12]:
# ============================================================
# 13. 最终测试集评估
# ============================================================
test_metrics = trainer.evaluate(test_dataset)

print("\nFinal test metrics:")
print(test_metrics)


Final test metrics:
{'eval_loss': 0.7967133522033691, 'eval_accuracy': 0.66574, 'eval_runtime': 191.4131, 'eval_samples_per_second': 261.215, 'eval_steps_per_second': 32.652, 'epoch': 3.0}


In [13]:
# ============================================================
# 14. 保存最佳模型、tokenizer 和训练历史
# ============================================================
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

stage1_checkpoint = (
    "models/bert-base-cased-finetune-yelp-full/"
    "checkpoint-238875"
)

stage1_model_dir = (
    "/sigan/ai-ml/models/finetuned/"
    "bert-base-cased-yelp-stage1"
)

model = AutoModelForSequenceClassification.from_pretrained(
    stage1_checkpoint
)
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-cased"
)

model.save_pretrained(stage1_model_dir)
tokenizer.save_pretrained(stage1_model_dir)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/tokenizer_config.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/special_tokens_map.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/vocab.txt',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/added_tokens.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/tokenizer.json')

### 判断
1. 根据结果看，还有继续提升的空间，再训练一轮，判断依据如下：
   | 第二阶段结果 | 动作 |
    |---|---|
| Accuracy > 67.17%，且提升 ≥ 1 个百分点 | 可以再跑 1 个 epoch |
| Accuracy 小幅提升但不到约 0.8 个百分点 | 视为接近平台，不必再跑 3 轮 |
| Accuracy 上升但 eval_loss 开始明显上升 | 已出现轻微过拟合，保留最佳 checkpoint |
| Accuracy 和 eval_loss 同时变差 | 立即停止，回到第一阶段模型 |
3. 这里有个操作失误，提前跑了测试集，学习阶段可以接受，实际开发可能导致反向传播，需要重新训练，切记切记

In [14]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

model = AutoModelForSequenceClassification.from_pretrained(
    stage1_model_dir
)

stage2_dir = (
    "/sigan/ai-ml/models/finetuned/"
    "bert-base-cased-yelp-stage2"
)

stage2_args = TrainingArguments(
    output_dir=stage2_dir,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    learning_rate=1e-5,
    num_train_epochs=1,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,

    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=100,
    bf16=True,
    report_to="none",
    seed=42,
)

trainer_stage2 = Trainer(
    model=model,
    args=stage2_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.001,
        )
    ],
)

In [15]:
trainer_stage2.train()
print(trainer_stage2.evaluate(eval_dataset))

Epoch,Training Loss,Validation Loss,Accuracy
0,0.683600,0.794580,0.673231


{'eval_loss': 0.794579803943634, 'eval_accuracy': 0.6732307692307692, 'eval_runtime': 46.2401, 'eval_samples_per_second': 281.141, 'eval_steps_per_second': 17.582, 'epoch': 1.0}


In [16]:
test_metrics = trainer.evaluate(test_dataset)

print("\nFinal test metrics:")
print(test_metrics)


Final test metrics:
{'eval_loss': 0.7967133522033691, 'eval_accuracy': 0.66574, 'eval_runtime': 189.2632, 'eval_samples_per_second': 264.182, 'eval_steps_per_second': 33.023, 'epoch': 3.0}


In [17]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

stage1_checkpoint = (
    "models/bert-base-cased-finetune-yelp-full/"
    "checkpoint-238875"
)

stage1_model_dir = (
    "/sigan/ai-ml/models/finetuned/"
    "bert-base-cased-yelp-stage1"
)

model = AutoModelForSequenceClassification.from_pretrained(
    stage1_checkpoint
)
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-cased"
)

model.save_pretrained(stage1_model_dir)
tokenizer.save_pretrained(stage1_model_dir)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/tokenizer_config.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/special_tokens_map.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/vocab.txt',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/added_tokens.json',
 '/sigan/ai-ml/models/finetuned/bert-base-cased-yelp-stage1/tokenizer.json')

### 重新加载并测试

In [18]:
import gc
import torch

# 先释放旧对象
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

In [19]:
# 模拟全新运行环境加载

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)

tokenizer = AutoTokenizer.from_pretrained(stage1_model_dir)
model = AutoModelForSequenceClassification.from_pretrained(
    stage1_model_dir
)

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,
    top_k=None,
)

In [20]:
label_names = dataset["train"].features["label"].names

In [22]:
print(len(label_names))
print(label_names)

5
['1 star', '2 star', '3 stars', '4 stars', '5 stars']


In [25]:
# 测试几条英文评论

texts = [
    "The food was excellent and the service was very friendly.",
    "I waited forty minutes and the meal was cold.",
    "The restaurant was okay, but it was too expensive.",
]

for text in texts:
    predictions = classifier(text, truncation=True, max_length=512)

    # 如果外层是 list of list，展平一层
    if predictions and isinstance(predictions[0], list):
        predictions = predictions[0]

    print("\n评论：", text)
    for item in predictions:
        label_text = item["label"]
        probability = item["score"]

        if label_text.startswith("LABEL_"):
            label_id = int(label_text.split("_")[-1])
            label_text = label_names[label_id]

        print(f"{label_text}: {probability:.4f}")


评论： The food was excellent and the service was very friendly.
5 stars: 0.6653
4 stars: 0.3199
3 stars: 0.0136
2 star: 0.0007
1 star: 0.0005

评论： I waited forty minutes and the meal was cold.
1 star: 0.8731
2 star: 0.1124
3 stars: 0.0099
5 stars: 0.0029
4 stars: 0.0017

评论： The restaurant was okay, but it was too expensive.
2 star: 0.6558
3 stars: 0.2970
1 star: 0.0422
4 stars: 0.0046
5 stars: 0.0004
